# Main Test

In [1]:
import functions
from datetime import datetime
import pandas as pd

type = 'rt'
inicio = '01.04.2026'
fim = ''
day_minus = 0

list_df, init = functions.df_charge(type,inicio, fim)
masterfile = functions.masterfile_data('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/barcodes_ooh/OOH Masterfile - Barcodes Ativos - 2025_07_15 Resumo.CSV')
master = functions.conjunto_principal(list_df,masterfile,csv_path=False,reprocessing=False) # Requisição não está respeitando o filtro de data
print('\n dados carregados \n')
# A API não respeita os limites de data, então preciso fazer um filtro no pós-normalização
master_transformed = functions.transform(master)

hora_atual = datetime.now().strftime("%Y.%m.%d")

# Filtro de data
init = datetime.strptime(init, "%d.%m.%Y %H:%M:%S").date()
#master_transformed = master_transformed.loc[(master_transformed.StartTime >= init)]
master_transformed = master_transformed.loc[(master_transformed.StartTime >= init) &
(master_transformed.StartTime <= master_transformed.StartTime.max() - pd.Timedelta(days=day_minus))]

hora_atual = datetime.now().strftime("%Y.%m.%d")

print(f'\n periodo dos dados da API {init} a {master_transformed.StartTime.max()} \n Data de hoje {hora_atual}')

# Principal Processamento
master_transformed = functions.determina_ato_dnc(master_transformed) 
print('POINT')
estruct_analise = functions.estruct_analise(master_transformed)

print('\n determinação de atos concluída \n')


# Carregamento dos dados de amostra viva
amostra_viva = functions.amostra_base('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/amostra_base/data_PROC_ams_202601.CSV')
estruct_Inicial = estruct_analise.copy()
# pós-processamento
functions.propor_semana_limpeza(estruct_analise)
functions.propor_semana_limpeza(estruct_analise)
estruct_pos_inicio = estruct_analise.copy()
functions.process_desc_index(estruct_analise)   
desc_index = estruct_analise.copy()
functions.determina_elegibilidade(estruct_analise)
desc_elegib = estruct_analise.copy()
functions.determina_progresso_indiv(estruct_analise)

print('\n pós processamento finalizadoS\n')

individuos = functions.carrega_individuos('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/individuos/data_PROC_individuos_202601.CSV')
print('\n obtenção da mortalidade')

# Geração da basefinal
group1, group2 = functions.dados_finais(master_transformed,estruct_analise,amostra_viva, individuos, reprocessing=False)




c:\Users\luiz.farias\prod\atm_ooh_br\functions.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['dmProject'] = row['surveyId']
c:\Users\luiz.farias\prod\atm_ooh_br\functions.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['reportId'] = row['reportId']
c:\Users\luiz.farias\prod\atm_ooh_br\functions.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axi

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/barcodes_ooh/OOH Masterfile - Barcodes Ativos - 2025_07_15 Resumo.CSV'

# Visão diária

In [ ]:
group1.rename(columns={'Elegível' : 'Elegível_total'}, inplace=True)

group1 = group1[['dt_fim','Elegível_total']]

group2
group2.rename(columns={'Elegível' : 'Elegível_f2f'}, inplace=True)

merged = group1.merge(group2, on= 'dt_fim', how = 'inner')

merged['Elegível_ottros_prov'] = merged['Elegível_total'] - merged['Elegível_f2f']
merged.dt_fim = pd.to_datetime(merged.dt_fim)

merged

In [ ]:
consolidado = pd.read_excel('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/historico_eelegibilidade/consolidado/elegibilidade_diária_OOH_202601.xlsx')

consolidado = consolidado.merge(merged, on = "dt_fim",how='left')

consolidado.Elegível_total_y = consolidado.Elegível_total_y.fillna(consolidado.Elegível_total_x)

consolidado.drop(columns=['Elegível_total_x'], inplace=True)

consolidado.loc[:, 'WeekNumber'] = 'Semana' + consolidado['dt_fim'].apply(lambda x: str(x.isocalendar()[1]))

consolidado.to_excel('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/historico_eelegibilidade/consolidado/elegibilidade_diária_OOH_202601.xlsx', index=False)

consolidado

# Main para Amostra Piloto

In [ ]:
import functions

In [ ]:
import functions
from datetime import datetime

type = 'rt'
inicio = '01.06.2026'
fim = ''
day_minus = 0

list_df, init = functions.df_charge(type,inicio, fim, 'report_info_ams_piloto')

c:\Users\70089581\OneDrive - Kantar\Área de Trabalho\Workspace\Projeto_OOH\functions.py:39: DtypeWarning: Columns (21,37,39,56,58,59,75,77,78,79,82,83,85,87,88,89,90,91,92,93,94,95,98,99,100,101,102,103,104,105,106,107,109,111,113,114,115,117,118,121,122,123,125,126,128,130,132,133,159,162,164,176,187,190,198,202,210,213,220,222,224,225,238,242,250,253,260,262,264,265,271,272,273,277,282,284,293,298,299,300,304,306,308,309,312,314,315,317,318,323,325,327,328,331,333,334,336,337,342,344,346,347,350,352,353,355,356,362,364,366,367,370,372,373,375,376,380,382,386,399,401,424,457,459,461,476,481,491,495,498,499,500,505,508,514,522,524,527,529,530,531,532,533,534,536,545,548,549,553,555,557,566,567,568,585,593,595,600,601,609,626,628,633,636,637,642,651,652,653,656,657,665,666,667,670,671,693,697,702,706,711,722,737,755,767,781,784,786,789,790,793,795,807,810,815,817,824,827,832,834,835,836,838,839,843,846,848,852,855,857,860,862,866,869,871,875,878,880,883,885,887,893,898,905,906,917,930,9

In [ ]:
(list_df['qmob_project46960_report112785_Report1'].shape == list_df['qmob_project46960_report113060_Report2'].shape == list_df['qmob_project46960_report113091_Report3'].shape == list_df['qmob_project46960_report113415_Report4'].shape)

True

In [ ]:
list_df['qmob_project46960_report112785_Report1'].shape

(3501, 1362)

In [ ]:
import pandas as pd

In [ ]:
src = ['EntryID','EntryType','Username','StartTime','TripRef',
'pMeioPedido','pBRCat_BAC','pBRCat_BNA','pBRCat_BPC',
'pBRCat_BQS','pBRCat_CBC','pBRCat_DBO','pBRCat_FF',
'pBRCat_IOG','pBRCat_Lati','pBRCat_LSA','pBRCat_PCO',
'pBRCat_PIZ','pBRCat_SAN','pBRCat_SDP','pBRCat_SOR',
'pCatEnvasado','pCatFastFood','pScanner']

existentes = list_df['qmob_project46960_report112785_Report1'].columns

de_para = pd.DataFrame({
    'Coluna': src,
    'Existe': [col in existentes for col in src]
})


In [ ]:
de_para

,Coluna,Existe
0,EntryID,False
1,EntryType,False
2,Username,True
3,StartTime,False
4,TripRef,True
5,pMeioPedido,False
6,pBRCat_BAC,True
7,pBRCat_BNA,True
8,pBRCat_BPC,True
9,pBRCat_BQS,True


In [ ]:
nn = ['pSurveyStart','mainMenu','pInicio','pApp','pDondeCompro','pDondeCompro_Delivery',
'pFormaCompra','pFormaCompra_Delivery','pAccesoCanal','pMomentoConsu','pMomentoConsu_Delivery',
'pPagoFF','pPagoOt','PrecioTotal','Q260','Q260B','Q261','Q267','pshopEnd','pItemStart']


In [ ]:
list_df['qmob_project46960_report112785_Report1'][~list_df['qmob_project46960_report112785_Report1']['pBRCat_CBC'].isna()]['pBRCat_CBC']

16      cat=217;vSubX=6
192     cat=217;vSubX=5
241             cat=792
308     cat=793;vSubX=1
395     cat=793;vSubX=1
400     cat=217;vSubX=6
507     cat=793;vSubX=1
552             cat=792
556     cat=217;vSubX=5
564             cat=792
573             cat=792
586             cat=792
626             cat=792
755             cat=792
814             cat=792
815             cat=792
830     cat=217;vSubX=6
938     cat=217;vSubX=6
1042            cat=792
1059            cat=792
1060            cat=792
1133    cat=793;vSubX=1
1328            cat=792
1331            cat=792
1358    cat=217;vSubX=6
1545            cat=792
1569            cat=792
1834    cat=217;vSubX=6
1835    cat=217;vSubX=6
1894    cat=217;vSubX=6
1938    cat=793;vSubX=1
1996            cat=792
2136            cat=792
2178            cat=792
2328    cat=217;vSubX=6
2350            cat=792
2442            cat=792
2667            cat=792
2683    cat=217;vSubX=6
2794            cat=792
2856    cat=217;vSubX=5
2895            

In [ ]:
list_df['qmob_project46960_report112785_Report1']['PrecioTotal'].value_counts()

PrecioTotal
20.00     56
12.00     44
10.00     38
15.00     31
25.00     30
          ..
63.30      1
31.00      1
128.00     1
33.00      1
18.99      1
Name: count, Length: 256, dtype: int64

In [ ]:
list_df['qmob_project46960_report113060_Report2'].columns


Index(['Entry ID', 'Entry Type', 'Username', 'Upload time', 'Start time',
       'End time', 'Device Timezone', 'Device Timezone offset',
       'Survey revision', 'Survey wave',
       ...
       'Q75_FF_A.3', 'Q75FF', 'pItemEnd', 'pPhotoFactura.1', 'pPhotoFactura.2',
       'pPhotoFactura.3', 'pSurveyEnd', 'dmProject', 'reportId', 'nmReport'],
      dtype='object', length=1362)

In [ ]:
list_df['qmob_project46960_report113091_Report3'].columns



Index(['Entry ID', 'Entry Type', 'Username', 'Upload time', 'Start time',
       'End time', 'Device Timezone', 'Device Timezone offset',
       'Survey revision', 'Survey wave',
       ...
       'Q75_FF_A.3', 'Q75FF', 'pItemEnd', 'pPhotoFactura.1', 'pPhotoFactura.2',
       'pPhotoFactura.3', 'pSurveyEnd', 'dmProject', 'reportId', 'nmReport'],
      dtype='object', length=1362)

In [ ]:
list_df['qmob_project46960_report113415_Report4'].columns

Index(['Entry ID', 'Entry Type', 'Username', 'Upload time', 'Start time',
       'End time', 'Device Timezone', 'Device Timezone offset',
       'Survey revision', 'Survey wave',
       ...
       'Q75_FF_A.3', 'Q75FF', 'pItemEnd', 'pPhotoFactura.1', 'pPhotoFactura.2',
       'pPhotoFactura.3', 'pSurveyEnd', 'dmProject', 'reportId', 'nmReport'],
      dtype='object', length=1362)

In [ ]:
import functions
from datetime import datetime
import pandas as pd

type = 'rt'
inicio = '01.04.2026'
fim = ''
day_minus = 0

list_df, init = functions.df_charge(type,inicio, fim, 'report_info_ams_piloto')
masterfile = functions.masterfile_data('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/barcodes_ooh/OOH Masterfile - Barcodes Ativos - 2025_07_15 Resumo.CSV')
master = functions.conjunto_principal(list_df,masterfile,csv_path=False,reprocessing=False) # Requisição não está respeitando o filtro de data
print('\n dados carregados \n')
# A API não respeita os limites de data, então preciso fazer um filtro no pós-normalização
master_transformed = functions.transform(master)

hora_atual = datetime.now().strftime("%Y.%m.%d")

# Filtro de data
init = datetime.strptime(init, "%d.%m.%Y %H:%M:%S").date()
#master_transformed = master_transformed.loc[(master_transformed.StartTime >= init)]
master_transformed = master_transformed.loc[(master_transformed.StartTime >= init) &
(master_transformed.StartTime <= master_transformed.StartTime.max() - pd.Timedelta(days=day_minus))]

hora_atual = datetime.now().strftime("%Y.%m.%d")

print(f'\n periodo dos dados da API {init} a {master_transformed.StartTime.max()} \n Data de hoje {hora_atual}')

# Principal Processamento
master_transformed = functions.determina_ato_dnc(master_transformed) 
print('POINT')
estruct_analise = functions.estruct_analise(master_transformed)

print('\n determinação de atos concluída \n')


# Carregamento dos dados de amostra viva
amostra_viva = functions.amostra_base('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/amostra_base/data_PROC_ams_202601.CSV')
estruct_Inicial = estruct_analise.copy()
# pós-processamento
functions.propor_semana_limpeza(estruct_analise)
functions.propor_semana_limpeza(estruct_analise)
estruct_pos_inicio = estruct_analise.copy()
functions.process_desc_index(estruct_analise)   
desc_index = estruct_analise.copy()
functions.determina_elegibilidade(estruct_analise)
desc_elegib = estruct_analise.copy()
functions.determina_progresso_indiv(estruct_analise)

print('\n pós processamento finalizadoS\n')

individuos = functions.carrega_individuos('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/individuos/data_PROC_individuos_202601.CSV')
print('\n obtenção da mortalidade')

# Geração da basefinal
group1, group2 = functions.dados_finais(master_transformed,estruct_analise,amostra_viva, individuos, reprocessing=False)




# Nova Regra Elegibilidade
- Registra información válida para OOH en 3 o más semanas, independientemente del tipo de reporte.
- Registra información válida para OOH en 2 semanas y presenta actos de compra OOH en ambas semanas → siempre es elegible.
- Registra información válida para OOH en 2 semanas, pero actos de compra OOH solo en 1 semana → será elegible únicamente si registra al menos 3 actos de compra OOH en total durante el período.

### Ato e Dnc

In [3]:
import functions
from datetime import datetime
import pandas as pd

type = 'rt'
inicio = '01.09.2026'
fim = ''
day_minus = 0

list_df, init = functions.df_charge(type,inicio, fim)
masterfile = functions.masterfile_data('C:/Users/luiz.farias/Numerator International/BKO - Documents/Report/Elegibilidade OOH/projeto_ooh/datalake/barcodes_ooh/OOH Masterfile - Barcodes Ativos - 2025_07_15 Resumo.CSV')
master = functions.conjunto_principal(list_df,masterfile,csv_path=False,reprocessing=False) # Requisição não está respeitando o filtro de data
print('/n dados carregados /n')
# A API não respeita os limites de data, então preciso fazer um filtro no pós-normalização
master_transformed = functions.transform_w(master)

hora_atual = datetime.now().strftime("%Y.%m.%d")

"""# Filtro de data
init = datetime.strptime(init, "%d.%m.%Y %H:%M:%S").date()
#master_transformed = master_transformed.loc[(master_transformed.StartTime >= init)]
master_transformed = master_transformed.loc[(master_transformed.StartTime >= init) &
(master_transformed.StartTime <= master_transformed.StartTime.max() - pd.Timedelta(days=day_minus))]

hora_atual = datetime.now().strftime("%Y.%m.%d")

master_transformed = functions.determina_ato_dnc(master_transformed) 

dt_inicio = master_transformed.StartTime.min()
dt_fim = master_transformed.StartTime.max()
"""


c:\Users\luiz.farias\OneDrive - Numerator International\Área de Trabalho\Workspace\PROD\wp_ooh_atm_amostra_viva_ooh\wp_ooh_atm_amostra_viva\functions.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['dmProject'] = row['surveyId']
c:\Users\luiz.farias\OneDrive - Numerator International\Área de Trabalho\Workspace\PROD\wp_ooh_atm_amostra_viva_ooh\wp_ooh_atm_amostra_viva\functions.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['reportId'] = row['reportId']
c:\Users\luiz.farias\OneDrive - Numerator Internati

/n dados carregados /n


'# Filtro de data\ninit = datetime.strptime(init, "%d.%m.%Y %H:%M:%S").date()\n#master_transformed = master_transformed.loc[(master_transformed.StartTime >= init)]\nmaster_transformed = master_transformed.loc[(master_transformed.StartTime >= init) &\n(master_transformed.StartTime <= master_transformed.StartTime.max() - pd.Timedelta(days=day_minus))]\n\nhora_atual = datetime.now().strftime("%Y.%m.%d")\n\nmaster_transformed = functions.determina_ato_dnc(master_transformed) \n\ndt_inicio = master_transformed.StartTime.min()\ndt_fim = master_transformed.StartTime.max()\n'

In [ ]:


estruct_analise_v2 = master_transformed[['Username','ATO','DNC', 'WeekNumber']].copy()

estruct_analise_v2['ATO'] = pd.to_numeric(
estruct_analise_v2['ATO'],
errors='coerce'
)

estruct_analise_v2['DNC'] = pd.to_numeric(
estruct_analise_v2['DNC'],
errors='coerce'
)

estruct_analise_v2['status_comp'] = (
estruct_analise_v2['ATO']
.fillna(estruct_analise_v2['DNC'])
.fillna(0)
.astype(int)
)
funil1 = (
estruct_analise_v2
.groupby(['Username','WeekNumber'])['status_comp']
.value_counts()
.unstack(fill_value=0)
.reset_index()
.rename(columns={
0: 'Sem Registro',
1: 'DNC',
2: 'ATO'
})

)
funil1 = funil1.loc[~(funil1.Username.isin(['brtest','brteste','brhansel','55000016']))]

funil1.columns.name = None


funil1['wAto'] = funil1['ATO'] > 0
funil1['wAto>3'] = funil1['ATO'] >= 3

# verifica se existe ATO na semana
tem_ato_semana = funil1.groupby(['Username','WeekNumber'])['ATO'].transform(lambda x: (x > 0).any())

# verifica se existe DNC na semana
tem_dnc_semana = funil1.groupby(['Username','WeekNumber'])['DNC'].transform(lambda x: (x > 0).any())

# Mutually Exclusive Event Aggregation
funil1['wDnc'] = (funil1['DNC'] > 0) & ~(tem_ato_semana & tem_dnc_semana)


KeyError: 'ATO'

In [7]:
import functions
import glob
caminho = 'C:/Users/luiz.farias/Numerator International/BKO - Documents/Report/Elegibilidade OOH/projeto_ooh/datalake/consolida ooh/Consolida de Individuos OOH_*.xlsx'
funil1 = functions.funil_1(master_transformed)
funil2 = functions.funil_2(funil1)
estruct_analise = functions.estruct_analise_v2(funil1)
arquivo = glob.glob(caminho)[0]  # pega o primeiro que encontrar
amostra_viva = functions.amostra_viva(arquivo, estruct_analise)

KeyError: 'ATO'

In [ ]:
amostra_viva

,IndividuoOOH,data_entrada,InterviewerCode,Nome Entrev.,ATO_W10,ATO_W11,ATO_W12,ATO_W9,DNC_W10,DNC_W11,DNC_W12,DNC_W9,Participacao
0,br0550000098-01,2024-12-30,21448,MNT-Maria do Socorro Sousa,6,2,0,0,0,0,0,0,True
1,br004167-00,2015-09-15,21448,MNT-Maria do Socorro Sousa,0,8,2,0,0,0,0,0,True
2,br480326-00,2023-06-12,21257,F2F-Viviane Rocha Santos,0,0,0,0,1,0,0,0,True
3,br478967-00,2023-05-26,957,F2F-Sirley Alves de Almeida,0,0,0,2,0,0,0,0,True
4,br000008-00,2018-06-15,2325,F2F-Michele Biase Severonico,0,0,0,0,0,0,0,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6909,0552192503-09,2025-07-30,2422,MNT-Sheila Barreto,0,0,0,0,0,0,0,0,False
6910,0552209036-01,2026-03-06,21518,OOH-MNT-Lauany,0,0,2,0,0,0,0,0,True
6911,0552209037-01,2026-03-06,0,Setor(sem Entrevistadora),0,0,0,0,0,0,0,0,False
6912,0552209113-01,2026-03-06,21523,OOH-R&S-Lilian Silva,0,0,0,0,0,0,0,0,False


### Funil1 de análise
Objetivo:

1. Identificar quantidade atos por semana do periodo analisado;
2. Classificar semanas que tiveram atos,
3. Classificar semanas que tiveram DNC;
$. Classificar semanas que tiveram mais 3 atos na mesma semana;

In [ ]:
def funil_1(master_transformed):
    estruct_analise_v2 = master_transformed[['Username','StartTime','ATO','DNC', 'WeekNumber']].copy()

    estruct_analise_v2['ATO'] = pd.to_numeric(
        estruct_analise_v2['ATO'],
        errors='coerce'
    )

    estruct_analise_v2['DNC'] = pd.to_numeric(
        estruct_analise_v2['DNC'],
        errors='coerce'
    )

    estruct_analise_v2['status_comp'] = (
        estruct_analise_v2['ATO']
        .fillna(estruct_analise_v2['DNC'])
        .fillna(0)
        .astype(int)
    )
    
    funil1 = (
        estruct_analise_v2
        .groupby(['Username','StartTime','WeekNumber'])['status_comp']
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
        .rename(columns={
            0: 'Sem Registro',
            1: 'DNC',
            2: 'ATO'
        })
        
    )

    funil1.columns.name = None

    funil1 = funil1.loc[~(funil1.Username.isin(['brtest','brteste','brhansel','55000016']))]
    
    funil1['wAto'] = funil1['ATO'] > 0
    funil1['wAto>3'] = funil1['ATO'] >= 3

    # verifica se existe ATO na semana
    tem_ato_semana = funil1.groupby(['Username','WeekNumber'])['ATO'].transform(lambda x: (x > 0).any())

    # verifica se existe DNC na semana
    tem_dnc_semana = funil1.groupby(['Username','WeekNumber'])['DNC'].transform(lambda x: (x > 0).any())

    # Mutually Exclusive Event Aggregation
    funil1['wDnc'] = (funil1['DNC'] > 0) & ~(tem_ato_semana & tem_dnc_semana)
        
    return funil1, estruct_analise_v2



### Funil2 de análise

Objetivo:
1. Somar DNC e ATOS do periodo analisado;
2. Somar Quantidades de semana com Atos;
3. Semanas com DNC;
4. Semanas com MAIS de 3 ATOS;


In [ ]:
def funil_2(funil1):
    funil2 = (
        funil1
        .groupby(['Username'], as_index=False)
        .agg({
            'Sem Registro': 'sum',
            'DNC': 'sum',
            'ATO': 'sum',
            'wAto': 'sum',
            'wDnc': 'sum',
            'wAto>3': 'sum'
        })
    )

    import numpy as np 

    condicoes = [
        (funil2['wAto'] >= 2),
        (funil2['wDnc'] >= 3),
        (funil2['wDnc'] >= 2) & (funil2['wAto>3'] >=1) 
    ]

    valores = [
        True,
        True,
        True
    ]

    funil2['Elegivel'] = np.select(condicoes, valores, default=False)

    return funil2

### Amostra Viva

In [ ]:
# Nova Estrutura de dados por semana (Antigo Estruct Analise)
def estruct_analise_v2(funil1):
    pv = (
        pd.pivot_table(
            funil1,
            index = ['Username'],
            columns= 'WeekNumber',
            values= ['ATO', 'DNC'],
            aggfunc= 'sum'
        )
        .reset_index()
    )
    # achata o MultiIndex
    pv.columns = [
        f"{col[0]}_W{col[1]}" if isinstance(col, tuple) else col
        for col in pv.columns
    ]

    pv.rename(columns={
        'Username_W':'Username'
                    }, inplace=True)
    
    return pv


In [ ]:
def amostra_viva(str, estruct_analise_v2):
    individuos = functions.carrega_individuos(str)
    bs_ind = individuos[['IndividuoOOH', 'data_entrada', 'InterviewerCode', 'Nome Entrev.']].copy()
    bs_ind.InterviewerCode = bs_ind.InterviewerCode.fillna(0).astype('int64')    


    amostra_viva = bs_ind.merge(
        estruct_analise_v2,
        left_on='IndividuoOOH',
        right_on='Username',
        how='left'
    )

    amostra_viva['Participacao'] = amostra_viva['Username'].notna()
    amostra_viva.drop(columns = 'Username', inplace = True)

    prefixos_validos = ('ATO_W', 'DNC_W')

    cols_semana = [c for c in amostra_viva.columns if c.startswith(prefixos_validos)]

    amostra_viva[cols_semana] = amostra_viva[cols_semana].fillna(0).astype('int64')

    return amostra_viva


#### Ana Risia
- segmentar os ids do fornecedor F2F;
- Subir uma base de espelho, e criar uma nova coluna de classificação;

In [ ]:


origin_ids = pd.read_excel('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/egj_ids/1000_Preallocated_OOH_28112025.xlsx')

amostra_viva['origin_egj'] = 'Amostra Anterior'
amostra_viva.loc[
amostra_viva['IndividuoOOH'].isin(origin_ids['ALIAS']),
'origin_egj'
] = 'Ana Rissia'

### Elegibilidade 

- Reordenando colunas

In [ ]:
# Inclusão da Elegibilidade
def elegibles(funil2, amostra_viva, estruct_analise, dt_inicio, dt_fim):
    bs_elg = funil2[['Username', 'DNC', 'ATO', 'Elegivel']].copy()
    bs_elg['total_registros'] = bs_elg['DNC'] + bs_elg['ATO']
    bs_elg

    amostra_elegivel = amostra_viva.merge(
        bs_elg,
        left_on = 'IndividuoOOH',
        right_on = 'Username',
        how = 'left'

    )


    print("Shape pivot Analisys: " , estruct_analise.shape)
    print("Shape amostra_viva Join Base: " , amostra_viva.shape)
    print("Shape amostra_elegivel Join Base: " , amostra_elegivel.shape)


    amostra_elegivel['dt_inicio'] = dt_inicio
    amostra_elegivel['dt_fim'] = dt_fim

    amostra_elegivel.DNC = amostra_elegivel.DNC.fillna(0).astype('int64')	
    amostra_elegivel.ATO = amostra_elegivel.ATO.fillna(0).astype('int64')
    amostra_elegivel.total_registros = amostra_elegivel.total_registros.fillna(0).astype('int64')

    
    origin_ids = pd.read_excel('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/egj_ids/1000_Preallocated_OOH_28112025.xlsx')

    amostra_viva['origin_egj'] = 'Amostra Anterior'
    amostra_viva.loc[
    amostra_viva['IndividuoOOH'].isin(origin_ids['ALIAS']),
    'origin_egj'
    ] = 'Ana Rissia'

    # Colunas que você quer trazer para o início
    cols_inicio = [
        'dt_inicio',
        'dt_fim',
        'IndividuoOOH', 
        'data_entrada', 
        'InterviewerCode', 
        'Nome Entrev.' 
        'Participacao', 
        'origin_egj'
    ]

    # Garante que só usa colunas que realmente existem no df
    cols_inicio = [c for c in cols_inicio if c in amostra_elegivel.columns]

    # Reordena: primeiro as escolhidas, depois o restante
    amostra_elegivel = amostra_elegivel[cols_inicio + [c for c in amostra_elegivel.columns if c not in cols_inicio]]

    return amostra_elegivel

### Data Export

In [ ]:
def final_ams(df, dt_inicio, dt_fim):
    
    from openpyxl import load_workbook
    from openpyxl.styles import PatternFill
    from openpyxl.formatting.rule import FormulaRule
    from openpyxl.utils import get_column_letter

    hora_atual = datetime.now().strftime("%H-%M-%S")
    sharepoint_path = f'C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/historico_eelegibilidade'
    excel_filename = f'{sharepoint_path}/Elegibilidade_NEW_RULE_OOH_{dt_inicio}_A_{dt_fim}_{hora_atual}.xlsx'

    # Salva primeiro
    df.to_excel(excel_filename, index=False)

    # Abre o arquivo salvo
    wb = load_workbook(excel_filename)
    ws = wb.active

    # Define cores
    verde_claro = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    amarelo_claro = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")

    # Descobre colunas dinamicamente
    colunas = list(df.columns)

    for idx, col in enumerate(colunas, start=1):
        
        if col.startswith("ATO_W") or col.startswith("DNC_W"):
            
            col_letter = get_column_letter(idx)
            range_col = f"{col_letter}2:{col_letter}{ws.max_row}"

            # Fórmula: célula > 0
            formula = f"{col_letter}2>0"

            if col.startswith("ATO_W"):
                rule = FormulaRule(formula=[formula], fill=verde_claro)
            else:
                rule = FormulaRule(formula=[formula], fill=amarelo_claro)

            ws.conditional_formatting.add(range_col, rule)

    # Salva novamente
    wb.save(excel_filename)
    ws = wb.close()

In [ ]:
individuos = functions.carrega_individuos('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/individuos/data_PROC_individuos_202602.CSV')

### Função Main (nova)

In [ ]:
import functions
from datetime import datetime
import pandas as pd
import glob

type = 'rt'
inicio = '01.04.2026'
fim = ''
day_minus = 0


list_df, init = functions.df_charge(type,inicio, fim)
masterfile = functions.masterfile_data('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/barcodes_ooh/OOH Masterfile - Barcodes Ativos - 2025_07_15 Resumo.CSV')
master = functions.conjunto_principal(list_df,masterfile,csv_path=False,reprocessing=False) # Requisição não está respeitando o filtro de data
print('\n dados carregados \n')
# A API não respeita os limites de data, então preciso fazer um filtro no pós-normalização
master_transformed = functions.transform_w(master)



# Filtro de data
init = datetime.strptime(init, "%d.%m.%Y %H:%M:%S").date()
#master_transformed = master_transformed.loc[(master_transformed.StartTime >= init)]
master_transformed = master_transformed.loc[(master_transformed.StartTime >= init) &
(master_transformed.StartTime <= master_transformed.StartTime.max() - pd.Timedelta(days=day_minus))]



master_transformed = functions.determina_ato_dnc(master_transformed) 

dt_inicio = master_transformed.StartTime.min()
dt_fim = master_transformed.StartTime.max()


funil1 = functions.funil_1(master_transformed)
funil2 = functions.funil_2(funil1)
estruct_analise = functions.estruct_analise_v2(funil1)
caminho = 'C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/consolida ooh/Consolida de Individuos OOH_*.xlsx'
arquivo = glob.glob(caminho)[0]  # pega o primeiro que encontrar
amostra_viva = functions.amostra_viva(arquivo, estruct_analise)
eleg = functions.elegibles(funil2, amostra_viva, estruct_analise, dt_inicio, dt_fim)

amostra_elegivel = functions.final_ams(eleg, dt_inicio, dt_fim)
functions.salvar_csv_sharepoint(master_transformed)


C:\Users\70089581\AppData\Roaming\Python\Python312\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.3.0)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
c:\Users\70089581\OneDrive - Kantar\Área de Trabalho\Workspace\Projeto_OOH\functions.py:39: DtypeWarning: Columns (6,8,13,14,23) have mixed types. Specify dtype option on import or set low_memory=False.
  dados = pd.read_csv(file_path)
c:\Users\70089581\OneDrive - Kantar\Área de Trabalho\Workspace\Projeto_OOH\functions.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([list_df['qmob_project39026_report67317_Report2'],



 dados carregados 

[2026-04-09 14:17:57.360067] Arquivo sobrescrito em: C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/actos_qmob/survey_2026-04-01.csv


In [ ]:
amostra_elegivel

,dt_inicio,dt_fim,IndividuoOOH,data_entrada,InterviewerCode,Provedor,Nome Entrev.,Região Kantar,Macro Região,ATO_W14,...,DNC,ATO,wAto,wDnc,wAto>3,total_registros,Elegivel_nr,Elegivel_nr_desc,Elegivel_ls,Progresso_individuo
0,2026-04-01,2026-04-09,br0550000098-01,2024-12-30,21448,Sem Provedor,MNT-Maria do Socorro Sousa,Centro-Oeste,Centro-Oeste,4,...,0,8,2.0,0.0,2.0,8,True,ATO > 2 semanas,True,já É um elegível
1,2026-04-01,2026-04-09,br004167-00,2015-09-15,21448,Sem Provedor,MNT-Maria do Socorro Sousa,RM SP,GSP,4,...,0,8,2.0,0.0,2.0,8,True,ATO > 2 semanas,True,já É um elegível
2,2026-04-01,2026-04-09,br480326-00,2023-06-12,21257,Sem Provedor,F2F-Viviane Rocha Santos,RM SP,GSP,0,...,2,0,0.0,2.0,0.0,2,False,False,False,Falta 1 ATO
3,2026-04-01,2026-04-09,br478967-00,2023-05-26,957,Sem Provedor,F2F-Sirley Alves de Almeida,RM BH,RM BH+Resto Leste+Int RJ,0,...,0,0,NaN,NaN,NaN,0,NaN,NaN,NaN,Sem Registro
4,2026-04-01,2026-04-09,br000008-00,2018-06-15,2325,Sem Provedor,F2F-Michele Biase Severonico,RM RJ,GRJ,0,...,0,0,NaN,NaN,NaN,0,NaN,NaN,NaN,Sem Registro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7271,2026-04-01,2026-04-09,0552219077-01,2026-03-18,21523,22 - TOP Client,OOH-R&S-Lilian Silva,RM REC,Nordeste,0,...,0,0,NaN,NaN,NaN,0,NaN,NaN,NaN,Sem Registro
7272,2026-04-01,2026-04-09,0552219078-01,2026-03-18,21518,22 - TOP Client,OOH-MNT-Lauany,RM SP,GSP,0,...,0,0,NaN,NaN,NaN,0,NaN,NaN,NaN,Sem Registro
7273,2026-04-01,2026-04-09,0552219082-01,2026-03-18,21537,22 - TOP Client,LUANA SANTANA,Interior São Paulo,Interior São Paulo,0,...,0,0,NaN,NaN,NaN,0,NaN,NaN,NaN,Sem Registro
7274,2026-04-01,2026-04-09,0552219123-01,2026-03-25,2195,22 - TOP Client,OOH-MNT-Josiclaudia,Norte,Resto N+NE,0,...,0,0,NaN,NaN,NaN,0,NaN,NaN,NaN,Sem Registro


In [ ]:
master_transformed.columns

Index(['EntryID', 'EntryType', 'Username', 'StartTime', 'TripRef',
       'pMeioPedido', 'pBRCat_BAC', 'pBRCat_BNA', 'pBRCat_BPC', 'pBRCat_BQS',
       'pBRCat_CBC', 'pBRCat_DBO', 'pBRCat_FF', 'pBRCat_IOG', 'pBRCat_Lati',
       'pBRCat_LSA', 'pBRCat_PCO', 'pBRCat_PIZ', 'pBRCat_SAN', 'pBRCat_SDP',
       'pBRCat_SOR', 'pCatEnvasado', 'pCatFastFood', 'pScanner', 'dmProject',
       'reportId', 'nmReport', 'CodBarra', 'IdArtigo', 'Sector', 'IdProd',
       'Producto', 'COLETADO NO OOH?', 'WeekNumber', 'ATO', 'DNC'],
      dtype='object')

In [ ]:
master_transformed.Username.nunique()

3329

In [ ]:
eleg.loc[eleg.Username.notna()].IndividuoOOH.nunique()

3142

In [ ]:
3328 - 3141

187

In [ ]:
ass = master_transformed.loc[master_transformed.Username.isin(eleg.IndividuoOOH)]
print(ass.shape)
print(master_transformed.shape)

(42335, 36)
(43459, 36)


In [ ]:
import glob 

caminho = 'C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/consolida ooh/Consolida de Individuos OOH_*.xlsx'
arquivo = glob.glob(caminho)[0]  # pega o primeiro que encontrar

In [ ]:
import numpy as np

master_transformed.ATO = master_transformed.ATO.fillna(0)
master_transformed.DNC = master_transformed.DNC.fillna(0)

master_transformed = master_transformed.loc[~(master_transformed.Username.isin(['55000016', 'brhansel']))]


master_transformed["Month_Ref"] = pd.to_datetime(master_transformed["StartTime"], format="%d/%m/%Y").dt.strftime("%Y-%m")

# Identificação de semana

# início do mês de cada linha
master_transformed['StartTime'] = pd.to_datetime(master_transformed['StartTime'])

month_start = master_transformed['StartTime'].dt.to_period('M').dt.start_time

# weekday: segunda=0 ... domingo=6
ms_wd = month_start.dt.weekday

# primeira segunda-feira *dentro* do mês
first_monday = month_start + pd.to_timedelta((7 - ms_wd) % 7, unit='D')

# início da semana (segunda) da master_transformed (pode cair no mês anterior)
week_start = master_transformed['StartTime'] - pd.to_timedelta(master_transformed['StartTime'].dt.weekday, unit='D')

# flag: mês começa na segunda?
starts_monday = (ms_wd == 0)

# semana_mes operacional (1,2,3...)
master_transformed['semana_mes'] = np.where(
    master_transformed['StartTime'] < first_monday,                         # antes da 1ª segunda do mês
    1,                                                        # => Semana 1 (curta)
    1 + (week_start - first_monday).dt.days // 7 + (~starts_monday).astype(int)
)
 

# rótulo bonitinho (1ª, 2ª, 3ª...)
master_transformed['Periodo'] = master_transformed['semana_mes'].map(lambda x: f"{x}ª Semana")
master_transformed['dia_mes'] = master_transformed['StartTime'].dt.day

# ---------------------------------------------------
pv_gran_dia = master_transformed.loc[
    ~((master_transformed.ATO == 0) & (master_transformed.DNC == 0))
    ].groupby(['StartTime','WeekNumber', 'semana_mes']).agg(
        ATO=('ATO', lambda x: np.sum(x == 2)),
        DNC=('DNC', lambda x: np.sum(x == 1)),
        Username=('Username', 'nunique')
                                                                                                                        
).reset_index()




C:\Users\70089581\AppData\Local\Temp\ipykernel_39204\1471858811.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_transformed.ATO = master_transformed.ATO.fillna(0)
C:\Users\70089581\AppData\Local\Temp\ipykernel_39204\1471858811.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_transformed.DNC = master_transformed.DNC.fillna(0)


In [ ]:
pivot = eleg.pivot_table(
    index='dt_fim',
    columns='Progresso_individuo',
    values='IndividuoOOH',
    aggfunc='nunique',
    fill_value=0
).reset_index()
pivot

Progresso_individuo,dt_fim,Falta 1 Ato,Falta 1 Dnc,Falta 2 Ato,Já é um elegível
0,2026-03-17,1048,216,270,1608


In [ ]:
pivot = eleg.pivot_table(
    index='dt_fim',
    columns='Progresso_individuo',
    values='IndividuoOOH',
    aggfunc='nunique',
    fill_value=0
).reset_index()

eleg_dia_d = eleg.groupby(['dt_fim']).agg(
    Elegiveis_ls=('Elegivel_nr', 'sum'),
    Elegiveis_nr=('Elegivel_ls', 'sum'),
    Amostra_viva=('IndividuoOOH', 'nunique')
).reset_index()

eleg_dia_d['dt_fim'] = pd.to_datetime(eleg_dia_d['dt_fim'])
eleg_dia_d = eleg_dia_d.merge(pivot, on='dt_fim', how='left')
eleg_dia_d

,dt_fim,Elegiveis_ls,Elegiveis_nr,Amostra_viva,Falta 1 Ato,Falta 1 Dnc,Falta 2 Ato,Já é um elegível
0,2026-03-17,1899,1859,6911,1048,216,270,1608


In [ ]:
append = eleg_dia_d.merge(pv_gran_dia, how='inner', left_on='dt_fim', right_on='StartTime')
append.rename(columns={"Username":"Participando_dia"},inplace=True)
append = append[['dt_fim','WeekNumber',	'semana_mes','Elegiveis_ls','Elegiveis_nr',	'Amostra_viva','Participando_dia','ATO','DNC']]
append

,dt_fim,WeekNumber,semana_mes,Elegiveis_ls,Elegiveis_nr,Amostra_viva,Participando_dia,ATO,DNC
0,2026-03-17,12,4,1899,1859,6911,727,1047,357


In [ ]:
eleg_dia_d.dtypes

dt_fim          datetime64[ns]
Elegiveis_ls            object
Elegiveis_nr            object
Amostra_viva             int64
dtype: object

In [ ]:
pv_gran_dia.dtypes

StartTime     datetime64[ns]
WeekNumber            object
semana_mes             int64
ATO                    int64
DNC                    int64
Username               int64
dtype: object

In [ ]:
funil1.loc[funil1.Username == 'br481490-00']

,Username,StartTime,WeekNumber,Sem Registro,DNC,ATO,wAto,wAto>3,wDnc
21186,br481490-00,2026-02-02,6,1,0,2,True,False,False
21187,br481490-00,2026-02-04,6,1,0,2,True,False,False


In [ ]:
hora_atual = datetime.now().strftime("%H-%M-%S")
hora_atual

'18-05-18'

# Reprocessamento


In [ ]:
def main(type, inicio, fim, day_minus):
    import functions
    from datetime import datetime
    import glob
    import pandas as pd

    list_df, init = functions.df_charge(type,inicio, fim)
    masterfile = functions.masterfile_data('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/barcodes_ooh/OOH Masterfile - Barcodes Ativos - 2025_07_15 Resumo.CSV')
    master = functions.conjunto_principal(list_df,masterfile,csv_path=False,reprocessing=False) # Requisição não está respeitando o filtro de data
    # A API não respeita os limites de data, então preciso fazer um filtro no pós-normalização
    master_transformed = functions.transform_w(master)

    

    # Filtro de data
    init = datetime.strptime(init, "%d.%m.%Y %H:%M:%S").date()
    #master_transformed = master_transformed.loc[(master_transformed.StartTime >= init)]
    master_transformed = master_transformed.loc[(master_transformed.StartTime >= init) &
    (master_transformed.StartTime <= master_transformed.StartTime.max() - pd.Timedelta(days=day_minus))]

    print(f'periodo: {init} a {master_transformed.StartTime.max()}')

    
    master_transformed = functions.determina_ato_dnc(master_transformed) 

    dt_inicio = master_transformed.StartTime.min()
    dt_fim = master_transformed.StartTime.max()
    

    functions.salvar_csv_sharepoint(master_transformed)



    

    funil1 = functions.funil_1(master_transformed)
    funil2 = functions.funil_2(funil1)
    estruct_analise = functions.estruct_analise_v2(funil1)

    
    
    caminho = 'C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/consolida ooh/Consolida de Individuos OOH_*.xlsx'
    arquivo = glob.glob(caminho)[0]  # pega o primeiro que encontrar
    amostra_viva = functions.amostra_viva(arquivo, estruct_analise)
    #amostra_viva = functions.amostra_viva('C:/Users/70089581/Kantar/BKO - projeto_ooh/datalake/individuos/data_PROC_individuos_202602.CSV', estruct_analise)

    
    eleg = functions.elegibles(funil2, amostra_viva, estruct_analise, dt_inicio, dt_fim)
    
    
    functions.final_ams(eleg, dt_inicio, dt_fim)




In [ ]:
from main_nr import main
i = 10
j = 22
data_abstract = f'01.{j-i}.2026'
while True:
    print(f'({data_abstract})day minus:  ', i)
    main('rt', "01.07.2026", '',i) 
    i = i - 1
    if i == 8:
        break    

(01.12.2026)day minus:   10


c:\Users\luiz.farias\prod\atm_ooh_br\functions.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['dmProject'] = row['surveyId']
c:\Users\luiz.farias\prod\atm_ooh_br\functions.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp['reportId'] = row['reportId']
c:\Users\luiz.farias\prod\atm_ooh_br\functions.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axi

KeyboardInterrupt: 

# Dados diários Last Rule

In [ ]:
def carregamento(prefixo, path):
    import pandas as pd
    import glob
    import os

    # Busca CSV e Excel
    arquivos = glob.glob(os.path.join(path, f"{prefixo}*.csv")) + \
               glob.glob(os.path.join(path, f"{prefixo}*.xlsx")) + \
               glob.glob(os.path.join(path, f"{prefixo}*.xls"))

    dfs = []

    for arq in arquivos:
        extensao = os.path.splitext(arq)[1].lower()

        if extensao == ".csv":
            df = pd.read_csv(arq, sep=';', low_memory=False)

        elif extensao in [".xlsx", ".xls"]:
            df = pd.read_excel(arq)

        else:
            continue  # ignora formatos desconhecidos

        dfs.append(df)

    if not dfs:
        raise ValueError("Nenhum arquivo encontrado para o prefixo informado.")

    df_final = pd.concat(dfs, ignore_index=True)

    return df_final

In [ ]:
ss = 'C:/Users/luiz.farias/Numerator International/BKO - projeto_ooh/datalake/historico_eelegibilidade'

eleg = carregamento(prefixo="Elegibilidade_NR_OOH",path = ss)

In [ ]:
eleg['semana_num'] = eleg['dt_fim'].dt.isocalendar().week

In [ ]:
df = eleg.groupby(['dt_fim','semana_num']).agg(
    qty = ('Elegivel_nr', 'sum')
).reset_index()

df.qty = df.qty.astype(int)

df['daily'] = df['qty'].diff()
df.daily = df.daily.fillna(0).astype(int)
from datetime import datetime
date = datetime.now().strftime('%Y%m')

ss = 'C:/Users/luiz.farias/Numerator International/BKO - projeto_ooh/datalake/historico_eelegibilidade/consolidado/'

df.to_excel(ss + f'elegibilidade_diária_OOH_{date}.xlsx', index=False)

In [ ]:
df

,dt_fim,semana_num,qty,daily
0,2026-06-02,23,0,0
1,2026-06-03,23,0,0
2,2026-06-04,23,0,0
3,2026-06-05,23,0,0
4,2026-06-06,23,0,0
5,2026-06-07,23,0,0
6,2026-06-08,24,179,179
7,2026-06-09,24,841,662
8,2026-06-10,24,1029,188
9,2026-06-11,24,1559,530


## Re-preocessamento